# Import Photometer Metadata to DB

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
import requests
import os

#### Definition DB

In [ ]:
# Define the database path
db_name = '../data/TessNetwork_metadata.db'

# Check if the database file exists
if not os.path.isfile(db_name):
    print("Database does not exist. Creating a new one.")
else:
    print("Database already exists.")

# Create the engine regardless, since it will just connect to the existing database
engine = create_engine(f'sqlite:///{db_name}')

# SQL query to drop the table if it exists
drop_table_query = "DROP TABLE IF EXISTS TessNetwork_metadata;"

# SQL query to create the table
create_table_query = '''
CREATE TABLE TessNetwork_metadata (
    name VARCHAR(255),
    latitude DECIMAL(10, 2),
    longitude DECIMAL(10, 2),
    country VARCHAR(255),
    city VARCHAR(255),
    place VARCHAR(255),
    local_timezone_original VARCHAR(50),
    local_timezone_mapping VARCHAR(50),
    org_name VARCHAR(255),
    org_web_url VARCHAR(255)
);
'''

# Connect to the database, drop the table, and then create a new one
with engine.connect() as connection:
    # Drop the existing table if it exists
    connection.execute(text(drop_table_query))
    print("Table dropped successfully (if it existed).")
    
    # Create the table
    connection.execute(text(create_table_query))
    print("Table created successfully.")

#### API Metadata to SQLite Database

In [ ]:
# API URL
api_url = "https://api.stars4all.eu/photometers"

# Fetch data from the API
response = requests.get(api_url)
data = response.json()

# Parse data and load it into a list of dictionaries
records = []
for item in data:
    record = {
        "name": item.get("name"),
        "latitude": round(float(item.get("latitude")), 2) if item.get("latitude") is not None else None,
        "longitude": round(float(item.get("longitude")), 2) if item.get("longitude") is not None else None,
        "country": item.get("country", item.get("info_location", {}).get("country")),
        "city": item.get("city", item.get("info_location", {}).get("town")),
        "place": item.get("place", item.get("info_location", {}).get("place")),
        "local_timezone_original": item.get("local_timezone", item.get("info_tess", {}).get("local_timezone")),
        "local_timezone_mapping": '',  # Default value
        "org_name": item.get("info_org", {}).get("name"),
        "org_web_url": item.get("info_org", {}).get("web_url")
    }
    records.append(record)

# Convert to DataFrame
df = pd.DataFrame(records)

# Insert new data into SQLite database
df.to_sql('TessNetwork_metadata', con=engine, if_exists='append', index=False)

print("New data inserted successfully!")

In [ ]:
# Read the table from the Wikipedia page
df_tz = pd.read_html("https://en.wikipedia.org/wiki/List_of_tz_database_time_zones")

In [ ]:

# Select the first table on the page
time_zone_table = df_tz[0]

# Extract the specific columns
timezone_mapping = time_zone_table.loc[:, [('TZ identifier', 'TZ identifier'), ('UTC offset ±hh:mm', 'SDT')]]

# Rename columns for simplicity
timezone_mapping.columns = ['TZ_identifier', 'UTC_offset_hh_mm']

# Display the result
print(timezone_mapping)


In [ ]:
import requests
import pandas as pd

# API URL
api_url = "https://api.stars4all.eu/photometers"

# Fetch data from the API
response = requests.get(api_url)
data = response.json()

# Parse data and load it into a list of dictionaries
records = []
for item in data:
    record = {
        "name": item.get("name"),
        "latitude": round(float(item.get("latitude")), 2) if item.get("latitude") is not None else None,
        "longitude": round(float(item.get("longitude")), 2) if item.get("longitude") is not None else None,
        "country": item.get("country", item.get("info_location", {}).get("country")),
        "city": item.get("city", item.get("info_location", {}).get("town")),
        "place": item.get("place", item.get("info_location", {}).get("place")),
        "local_timezone_original": item.get("local_timezone", item.get("info_tess", {}).get("local_timezone")),
        "local_timezone_mapping": '',  # Placeholder for the mapped value
        "org_name": item.get("info_org", {}).get("name"),
        "org_web_url": item.get("info_org", {}).get("web_url")
    }
    records.append(record)

# Convert records to DataFrame
df = pd.DataFrame(records)

# Assuming `df_tz[0]` is the DataFrame containing the timezone mapping
time_zone_table = df_tz[0]

# Extract the relevant columns from `time_zone_table` into the `timezone_mapping` DataFrame
timezone_mapping = time_zone_table.loc[:, [('TZ identifier', 'TZ identifier'), ('UTC offset ±hh:mm', 'SDT')]]

# Rename columns for simplicity
timezone_mapping.columns = ['TZ_identifier', 'UTC_offset_hh_mm']

# Merge the timezone mapping with the original data
df = pd.merge(df, timezone_mapping, left_on='local_timezone_original', right_on='TZ_identifier', how='left')

# Drop the 'TZ_identifier' column if no longer needed
df.drop(columns=['TZ_identifier'], inplace=True)

# Insert the enriched data into the SQLite database
df.to_sql('TessNetwork_metadata', con=engine, if_exists='append', index=False)

print("New data inserted successfully!")


### Versuch Nando

In [2]:
# 1. Import Libraries
import os
import sqlite3
import requests
import pandas as pd
from sqlalchemy import create_engine

# 2. Define and Create Database Path
db_path = '../data/'
db_name = os.path.join(db_path, 'TessNetwork_metadata.db')
os.makedirs(db_path, exist_ok=True)

# Create or Recreate Database and Table
if not os.path.exists(db_name):
    with sqlite3.connect(db_name) as conn:
        conn.execute("""
        CREATE TABLE TessNetwork_metadata (
            name VARCHAR(255),
            latitude DECIMAL(10, 2),
            longitude DECIMAL(10, 2),
            country VARCHAR(255),
            city VARCHAR(255),
            place VARCHAR(255),
            local_timezone_name VARCHAR(50),
            local_timezone VARCHAR(50),
            org_name VARCHAR(255)
        );
        """)
else:
    with sqlite3.connect(db_name) as conn:
        conn.execute("DROP TABLE IF EXISTS TessNetwork_metadata;")
        conn.execute("""
        CREATE TABLE TessNetwork_metadata (
            name VARCHAR(255),
            latitude DECIMAL(10, 2),
            longitude DECIMAL(10, 2),
            country VARCHAR(255),
            city VARCHAR(255),
            place VARCHAR(255),
            local_timezone_name VARCHAR(50),
            local_timezone VARCHAR(50),
            org_name VARCHAR(255)
        );
        """)

engine = create_engine(f'sqlite:///{db_name}')

# 3. Fetch API Data for Photometer Metadata
photometer_api_url = "http://api.stars4all.eu/photometers"

def fetch_photometer_metadata(api_url):
    response = requests.get(api_url)
    response.raise_for_status()
    data = response.json()

    # Debug: Inspect the raw API response
    print("Raw API Response (first 3 records):")
    print(data[:3])

    if not isinstance(data, list):
        raise ValueError("Expected API to return a list of photometer data.")

    # Process records
    records = []
    for item in data:
        record = {
            "name": item.get("name", ""),
            "latitude": round(item.get("info_location", {}).get("latitude", 0), 4) if item.get("info_location", {}).get("latitude") else None,
            "longitude": round(item.get("info_location", {}).get("longitude", 0), 4) if item.get("info_location", {}).get("longitude") else None,
            "country": item.get("info_location", {}).get("country", ""),
            "city": item.get("info_location", {}).get("town", ""),
            "place": item.get("info_location", {}).get("place", ""),
            "local_timezone_name": item.get("info_tess", {}).get("local_timezone", ""),
            "org_name": item.get("info_org", {}).get("name", ""),
        }
        records.append(record)

    df = pd.DataFrame(records)

    # Debug: Check the processed DataFrame
    print("Processed DataFrame (first 5 rows):")
    print(df.head())

    return df

photometer_metadata = fetch_photometer_metadata(photometer_api_url)
print(f"Total rows fetched from API: {len(photometer_metadata)}")

# 4. Map Timezones
wikipedia_url = "https://en.wikipedia.org/wiki/List_of_tz_database_time_zones"

def fetch_timezone_data(wikipedia_url):
    tables = pd.read_html(wikipedia_url, header=[0, 1])  # Fetch all tables with multi-level headers

    # Debug: Inspect all fetched tables
    print("Available Tables from Wikipedia:")
    for i, table in enumerate(tables):
        print(f"Table {i} Columns:")
        print(table.columns)

    for i, table in enumerate(tables):
        if ('TZ identifier' in table.columns.get_level_values(1) and 
            'UTC offset ±hh:mm' in table.columns.get_level_values(0)):
            # Extract the relevant columns
            timezone_data = table[[('TZ identifier', 'TZ identifier'), 
                                   ('UTC offset ±hh:mm', 'SDT')]].copy()
            timezone_data.columns = ['timezone', 'utc_offset']
            
            # Debug: Check the processed timezone DataFrame
            print("Timezone DataFrame Preview:")
            print(timezone_data.head())
            return timezone_data

    raise ValueError("Expected timezone table not found.")

timezone_data = fetch_timezone_data(wikipedia_url)

def map_timezones(photometer_df, timezone_df):
    def map_timezone(row):
        if not isinstance(row['local_timezone_name'], str) or not row['local_timezone_name']:
            return None  # Skip invalid or missing values
        if row['local_timezone_name'] in timezone_df['timezone'].values:
            return timezone_df.loc[timezone_df['timezone'] == row['local_timezone_name'], 'utc_offset'].values[0]
        if "UTC" in row['local_timezone_name']:
            try:
                offset = row['local_timezone_name'].replace("UTC", "").replace("+", "+0").replace("-", "-0")
                if len(offset) == 2:
                    offset += ":00"
                return offset
            except Exception:
                return None
        return None

    photometer_df['local_timezone'] = photometer_df.apply(map_timezone, axis=1)
    
    # Debug: Inspect the DataFrame after mapping
    print("Photometer DataFrame with Mapped Timezones:")
    print(photometer_df[['name', 'local_timezone_name', 'local_timezone']].head())
    
    return photometer_df

photometer_metadata = map_timezones(photometer_metadata, timezone_data)

# 5. Save to SQLite
photometer_metadata.to_sql('TessNetwork_metadata', con=engine, if_exists='replace', index=False)
print(f"Data successfully processed and saved to the database at '{db_name}'.")


Raw API Response (first 3 records):
[{'name': 'stars2', 'zero_point': 20.5, 'filters': 'UV/IR-cut', 'latitude': 41.0012989, 'longitude': -2.492408, 'country': 'España', 'city': 'Villaverde del Ducado', 'place': 'Villaverde', 'mov_sta_position': 'Estacionario', 'local_timezone': 'UTC+1', 'tester': 'Jaime Zamorano', 'location': 'Villaverde', 'info_img': {'urls': []}, 'info_tess': {'zero_point': 20.5, 'filters': 'UV/IR-740', 'period': 60, 'local_timezone': 'Europe/Madrid'}, 'info_location': {'longitude': -2.492408, 'latitude': 41.0013, 'place': 'Villaverde del Ducado', 'town': 'Villaverde del Ducado', 'sub_region': 'Guadalajara', 'region': 'Castile-La Mancha', 'country': 'Spain'}, 'info_org': {}}, {'name': 'stars5', 'zero_point': 20.38, 'filters': 'UV/IR-cut', 'latitude': 40.45098, 'longitude': -3.726072, 'country': 'España', 'city': 'Madrid', 'place': 'UCM', 'mov_sta_position': 'Estacionario', 'local_timezone': 'UTC+2', 'tester': 'Carlos Tapia', 'location': 'UCM', 'info_location': {'long